In [ ]:
"""
A script to fetch the loci in a search query from the ANTARES API.
"""

#from antares_client.search import search

QUERY = {
    "query": {
        "bool": {
            "filter": {
                "bool": {
                    "must": [
                        {
                            "terms": {
                                "tags": [
                                    "lantern_xgboost_t2.0.7_c0.95"
                                ]
                            }
                        },
                        {
                            "exists": {
                                "field": "properties.survey.lsst"
                            }
                        }
                    ]
                }
            }
        }
    }
}


#def process_locus(locus):
#    """Put your custom processing logic here."""
#    print(locus.locus_id)


#def main():
#    for locus in search(QUERY):
#        process_locus(locus)


#if __name__ == "__main__":
#    main()



In [ ]:
%%time

import json
import time
import pyarrow as pa
import pyarrow.parquet as pq

from pathlib import Path
from itertools import islice
from concurrent.futures import ThreadPoolExecutor, as_completed

from antares_client.search import search


# ============================================================
# Settings
# ============================================================

outdir = Path("antares_data")
locus_dir = outdir / "loci"
alert_dir = outdir / "alerts"

locus_dir.mkdir(parents=True, exist_ok=True)
alert_dir.mkdir(parents=True, exist_ok=True)

# Memory / disk batching
alert_batch_size = 100_000
locus_batch_size = 5_000

# Network concurrency
max_workers = 8

# Number of loci submitted at a time
fetch_batch_size = 32

# Test with first 10,000 loci
max_loci = 10_000

# Number of attempts for ANTARES API fetches
n_retry = 3


# ============================================================
# Parquet schemas
# ============================================================

locus_schema = pa.schema([
    ("locus_id", pa.string()),
    ("ra", pa.float64()),
    ("dec", pa.float64()),

    ("tags", pa.list_(pa.string())),
    ("catalogs", pa.list_(pa.string())),

    ("num_tagged_alerts", pa.int64()),
    ("max_score", pa.float64()),
    ("ztf_object_id", pa.string()),

    ("survey_lsst", pa.string()),

    ("gaia_variability_class", pa.list_(pa.string())),

    ("gaia_parallax_over_error", pa.list_(pa.float64())),
    ("gaia_pmdec", pa.list_(pa.float64())),
    ("gaia_pmdec_error", pa.list_(pa.float64())),
    ("gaia_pmra", pa.list_(pa.float64())),
    ("gaia_pmra_error", pa.list_(pa.float64())),
    ("gaia_in_qso_candidates", pa.list_(pa.bool_())),
])


alert_schema = pa.schema([
    ("locus_id", pa.string()),
    ("alert_id", pa.string()),
    ("mjd", pa.float64()),
    ("alert_properties", pa.string()),
])


# ============================================================
# Helpers
# ============================================================

def compact_json(x):

    if x is None:
        return None

    return json.dumps(
        x,
        separators=(",", ":"),
        default=str,
    )


def to_float(x):

    if x is None:
        return None

    return float(x)


def to_bool(x):

    if x is None:
        return None

    if isinstance(x, bool):
        return x

    if isinstance(x, str):

        x = x.strip().lower()

        if x == "true":
            return True

        if x == "false":
            return False

    return bool(x)


def fetch_with_retry(func, label):

    for attempt in range(n_retry):

        try:
            return func()

        except Exception as e:

            # Last attempt failed
            if attempt == n_retry - 1:
                raise

            delay = 2 ** attempt

            print(
                f"Retrying {label} in {delay} s "
                f"(attempt {attempt + 2}/{n_retry})"
            )

            time.sleep(delay)


def write_parquet(rows, schema, filename):

    table = pa.Table.from_pylist(
        rows,
        schema=schema,
    )

    pq.write_table(
        table,
        filename,
        compression="zstd",
    )


# ============================================================
# Process one locus
# ============================================================

def process_locus(locus):

    props = locus.properties or {}

    catalogs = list(locus.catalogs or [])
    catalog_set = set(catalogs)


    # --------------------------------------------------------
    # LSST information under locus.properties["survey"]
    # --------------------------------------------------------

    survey = props.get("survey", {})

    if isinstance(survey, dict):
        survey_lsst = survey.get("lsst")
    else:
        survey_lsst = None


    # --------------------------------------------------------
    # Gaia information
    # --------------------------------------------------------

    gaia_variability_class = []

    gaia_parallax_over_error = []
    gaia_pmdec = []
    gaia_pmdec_error = []
    gaia_pmra = []
    gaia_pmra_error = []
    gaia_in_qso_candidates = []


    # Only fetch catalog_objects if one of the Gaia catalogs
    # we care about is present.
    if (
        "gaia_dr3_variability" in catalog_set
        or "gaia_dr3_gaia_source" in catalog_set
    ):

        catalog_objects = fetch_with_retry(
            lambda: locus.catalog_objects,
            f"catalog_objects for {locus.locus_id}",
        ) or {}


        # ----------------------------------------------------
        # Gaia DR3 variability
        # ----------------------------------------------------

        for obj in catalog_objects.get(
            "gaia_dr3_variability", []
        ):

            value = obj.get("class")

            gaia_variability_class.append(
                str(value) if value is not None else None
            )


        # ----------------------------------------------------
        # Gaia DR3 Gaia source
        # ----------------------------------------------------

        for obj in catalog_objects.get(
            "gaia_dr3_gaia_source", []
        ):

            gaia_parallax_over_error.append(
                to_float(
                    obj.get("parallax_over_error")
                )
            )

            gaia_pmdec.append(
                to_float(
                    obj.get("pmdec")
                )
            )

            gaia_pmdec_error.append(
                to_float(
                    obj.get("pmdec_error")
                )
            )

            gaia_pmra.append(
                to_float(
                    obj.get("pmra")
                )
            )

            gaia_pmra_error.append(
                to_float(
                    obj.get("pmra_error")
                )
            )

            gaia_in_qso_candidates.append(
                to_bool(
                    obj.get("in_qso_candidates")
                )
            )


    # --------------------------------------------------------
    # Locus row
    # --------------------------------------------------------

    locus_row = {
        "locus_id": locus.locus_id,
        "ra": locus.ra,
        "dec": locus.dec,

        "tags": list(locus.tags or []),
        "catalogs": catalogs,

        "num_tagged_alerts": props.get(
            "lantern_xgboost_t2.0.7_c0.95_num_tagged_alerts"
        ),

        "max_score": props.get(
            "lantern_xgboost_t2.0.7_c0.95_max_score"
        ),

        "ztf_object_id": (
            str(props["ztf_object_id"])
            if props.get("ztf_object_id") is not None
            else None
        ),

        "survey_lsst": compact_json(
            survey_lsst
        ),

        "gaia_variability_class":
            gaia_variability_class,

        "gaia_parallax_over_error":
            gaia_parallax_over_error,

        "gaia_pmdec":
            gaia_pmdec,

        "gaia_pmdec_error":
            gaia_pmdec_error,

        "gaia_pmra":
            gaia_pmra,

        "gaia_pmra_error":
            gaia_pmra_error,

        "gaia_in_qso_candidates":
            gaia_in_qso_candidates,
    }


    # --------------------------------------------------------
    # Get alerts, with retry
    # --------------------------------------------------------

    alerts = fetch_with_retry(
        lambda: locus.alerts,
        f"alerts for {locus.locus_id}",
    )


    # --------------------------------------------------------
    # Keep only LSST alerts
    # --------------------------------------------------------

    alert_rows = []

    n_total = 0
    n_saved = 0
    n_skipped = 0


    for alert in alerts:

        n_total += 1


        if not alert.alert_id.startswith("lsst:"):

            n_skipped += 1
            continue


        alert_rows.append({
            "locus_id": locus.locus_id,
            "alert_id": alert.alert_id,
            "mjd": alert.mjd,

            # Keep ALL LSST alert properties
            "alert_properties": compact_json(
                alert.properties
            ),
        })

        n_saved += 1


    return (
        locus_row,
        alert_rows,
        n_total,
        n_saved,
        n_skipped,
    )


# ============================================================
# Storage / counters
# ============================================================

locus_rows = []
alert_rows = []

locus_part = 0
alert_part = 0

n_loci_processed = 0
n_loci_failed = 0

n_alerts_total = 0
n_alerts_saved = 0
n_non_lsst_skipped = 0

failed_loci = []


# ============================================================
# ANTARES search
# ============================================================

locus_iterator = iter(
    islice(
        search(QUERY),
        max_loci,
    )
)


# ============================================================
# Parallel processing
# ============================================================

with ThreadPoolExecutor(
    max_workers=max_workers
) as executor:

    while True:

        # ----------------------------------------------------
        # Get next batch of loci
        # ----------------------------------------------------

        batch = list(
            islice(
                locus_iterator,
                fetch_batch_size,
            )
        )

        if not batch:
            break


        # ----------------------------------------------------
        # Process loci in parallel
        # ----------------------------------------------------

        futures = {
            executor.submit(
                process_locus,
                locus,
            ): locus.locus_id
            for locus in batch
        }


        for future in as_completed(futures):

            locus_id = futures[future]


            # ------------------------------------------------
            # If a locus still fails after retries,
            # skip it and continue the run.
            # ------------------------------------------------

            try:

                (
                    locus_row,
                    alert_rows_local,
                    n_total,
                    n_saved,
                    n_skipped,
                ) = future.result()


            except Exception as e:

                n_loci_failed += 1
                failed_loci.append(locus_id)

                print(
                    f"FAILED locus {locus_id}: {e}"
                )

                continue


            # ------------------------------------------------
            # Successful locus
            # ------------------------------------------------

            locus_rows.append(
                locus_row
            )

            alert_rows.extend(
                alert_rows_local
            )

            n_loci_processed += 1

            n_alerts_total += n_total
            n_alerts_saved += n_saved
            n_non_lsst_skipped += n_skipped


            # ------------------------------------------------
            # Save alert batch
            # ------------------------------------------------

            if len(alert_rows) >= alert_batch_size:

                filename = (
                    alert_dir
                    / f"alerts_{alert_part:05d}.parquet"
                )

                write_parquet(
                    alert_rows,
                    alert_schema,
                    filename,
                )

                print(
                    f"Saved alert part {alert_part:05d}: "
                    f"{len(alert_rows):,} alerts"
                )

                alert_rows = []
                alert_part += 1


            # ------------------------------------------------
            # Save locus batch
            # ------------------------------------------------

            if len(locus_rows) >= locus_batch_size:

                filename = (
                    locus_dir
                    / f"loci_{locus_part:05d}.parquet"
                )

                write_parquet(
                    locus_rows,
                    locus_schema,
                    filename,
                )

                print(
                    f"Saved locus part {locus_part:05d}: "
                    f"{len(locus_rows):,} loci"
                )

                locus_rows = []
                locus_part += 1


        # ----------------------------------------------------
        # Progress after each batch of 32
        # ----------------------------------------------------

        print(
            f"Processed {n_loci_processed:,} loci total | "
            f"failed: {n_loci_failed:,} | "
            f"LSST alerts saved: {n_alerts_saved:,} | "
            f"non-LSST skipped: {n_non_lsst_skipped:,}"
        )


# ============================================================
# Save remaining data
# ============================================================

if alert_rows:

    write_parquet(
        alert_rows,
        alert_schema,
        alert_dir / f"alerts_{alert_part:05d}.parquet",
    )

    print(
        f"Saved final alert part {alert_part:05d}: "
        f"{len(alert_rows):,} alerts"
    )


if locus_rows:

    write_parquet(
        locus_rows,
        locus_schema,
        locus_dir / f"loci_{locus_part:05d}.parquet",
    )

    print(
        f"Saved final locus part {locus_part:05d}: "
        f"{len(locus_rows):,} loci"
    )


# ============================================================
# Summary
# ============================================================

print()
print("Finished")
print(f"Loci processed successfully: {n_loci_processed:,}")
print(f"Loci failed:                 {n_loci_failed:,}")
print(f"All alerts encountered:      {n_alerts_total:,}")
print(f"LSST alerts saved:           {n_alerts_saved:,}")
print(f"Non-LSST alerts skipped:     {n_non_lsst_skipped:,}")


if failed_loci:

    print()
    print("Failed locus IDs:")

    for locus_id in failed_loci:
        print(locus_id)